# Writing a workflow for the framework: specs and bindings

This notebook is for the authors of workflow packages, such as esssans.
A package that offers a workflow to the framework writes two things, both against `ess.spec`:

- a **spec**: the workflow's name, version, parameters, and outputs. Apps build forms from it, the framework checks requests against it, and records name it. It is all that anyone outside the package sees.
- a **binding**: the code that computes the spec. Apart from the validators of the params model, it is the only code of the package that the backend runs.

`ess.spec` needs only pydantic; `ess.spec.pipeline` adds sciline.
A workflow package depends on `ess.spec`, not on `ess.dispatch`, the client and backend that run it.

The thread: a reduction of one run, its spec and binding, then the same reduction over several runs, then summing runs as they arrive.
At each step, what the framework gives the binding, and what the binding's author promises in return.

## 0. The workflow you have: a sciline pipeline

The toy SANS reduction of `toy_sans.py` stands in for esssans.
A run's data holds its counts per pixel and its monitor count.
`load` stands for loading and correcting a run, the costly step.
The reduction sums the pixels into Q bins (the numerator), divides by the monitor count (the denominator), and subtracts the can's curve from the sample's.

In [1]:
import toy_sans
from toy_sans import (
    CAN_RUNS,
    SAMPLE_RUNS,
    Bins,
    CanRun,
    RunType,
    Counts,
    Denominator,
    IofQ,
    Numerator,
    RunData,
    SampleRun,
    pipeline,
)

print(SAMPLE_RUNS[101], CAN_RUNS[110])
workflow = pipeline.copy()
workflow[RunData[SampleRun]] = SAMPLE_RUNS[101]
workflow[RunData[CanRun]] = CAN_RUNS[110]
workflow[Bins] = 2
workflow.compute(IofQ)

{'counts': [8.0, 6.0, 4.0, 2.0], 'monitor': 10.0} {'counts': [1.0, 1.0, 1.0, 1.0], 'monitor': 10.0}


<scipp.Variable> (Q: 2)    float64  [dimensionless]  [1.2, 0.4]

## 1. The spec of one run

The spec declares the parameters as one pydantic model and the outputs as another.
Each field of the params model is a *parameter*.
A parameter is a *group*, a flat model whose fields are values or data, never another model, or a *table* of such models (section 4).

In [2]:
from pydantic import BaseModel, Field

from ess.spec import Array, ArraySpec, NexusFile, WorkflowSpec

CURVE = ArraySpec(dims=('Q',), unit='dimensionless')


class Run(BaseModel):
    file: NexusFile  # a data field: a raw NeXus file


class Reduction(BaseModel):
    bins: int = Field(default=2, gt=0)


class IofQParams(BaseModel):
    sample: Run
    can: Run
    reduction: Reduction = Field(default_factory=Reduction)


class IofQOutputs(BaseModel):
    iofq: Array(CURVE)  # a data field: a scipp array of this structure


IOFQ = WorkflowSpec(
    name='toy-sans-iofq',
    version=1,
    title='I(Q)',
    description='I(Q) of one sample run, minus that of one can run.',
    params=IofQParams,
    outputs=IofQOutputs,
)

Rules of the spec, each with its reason:

- **A data field holds data, given by reference.** A request names a dataset or another record's output; the binding gets the data read. So the spec says what the data is (`NexusFile`, `Array(...)`), never where it lives.
- **Defaults live in the params model only.** The binding gets every value filled in, so a default in the binding would never be used.
- **A check of one field is a field validator; a check of several is a model validator.** A template checks each value by its own field, before the others are known ([README, Templates](../README.md#templates)).
- **Name and version identify the spec.** Records name them. `version` is the version of the interface: it changes with any change a consumer could observe, in the params model, the outputs, or what the workflow computes. The optional field `code_revision` names the code itself; records do not carry it yet.
- **An `ArraySpec` describes the structure of an array.** Where both an output and the input it feeds declare one, the backend requires them to be equal when a request references the output. Nothing checks the computed value against it.

An app reads the spec, not the package's code: `serialize()` gives a JSON Schema from which a form is built.

In [3]:
schema = IOFQ.serialize().params_schema
{name: field.get('$ref', field) for name, field in schema['properties'].items()}

{'sample': '#/$defs/Run',
 'can': '#/$defs/Run',
 'reduction': '#/$defs/Reduction'}

The params model refuses what cannot run, before anything runs:

In [4]:
from pydantic import ValidationError

from ess.spec import DatasetRef

run = DatasetRef(dataset='run:101')
try:
    IofQParams(sample={'file': run}, can={'file': run}, reduction={'bins': 0})
except ValidationError as error:
    print(error)

1 validation error for IofQParams
reduction.bins
  Input should be greater than 0 [type=greater_than, input_value=0, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/greater_than


## 2. The binding of one run

The simplest binding is a function.
It takes the parameters by name and returns the outputs by name.
It gets each group as a dict of its fields, with every default filled in and every data field read:

In [5]:
def iofq(sample: dict, can: dict, reduction: dict) -> dict:
    workflow = pipeline.copy()
    workflow[RunData[SampleRun]] = sample['file']
    workflow[RunData[CanRun]] = can['file']
    workflow[Bins] = reduction['bins']
    return {'iofq': workflow.compute(IofQ)}


values = {
    'sample': {'file': SAMPLE_RUNS[101]},
    'can': {'file': CAN_RUNS[110]},
    'reduction': {'bins': 2},
}
iofq(**values)

{'iofq': <scipp.Variable> (Q: 2)    float64  [dimensionless]  [1.2, 0.4]}

For a sciline pipeline, `PipelineBinding` does the same from a table of names: the key that each field of a group sets, and the key that computes each output.

In [6]:
from ess.spec.pipeline import PipelineBinding

single = PipelineBinding(
    pipeline,
    params={
        'sample.file': RunData[SampleRun],
        'can.file': RunData[CanRun],
        'reduction.bins': Bins,
    },
    outputs={'iofq': IofQ},
)

The framework calls every binding in two steps, `stage(fixed, blanks)` and then the callable it returns.
A request with every value given, a *plain request*, has no blanks:

In [7]:
single.stage(values, ())()

{'iofq': <scipp.Variable> (Q: 2)    float64  [dimensionless]  [1.2, 0.4]}

**Promise: a binding modifies none of its inputs, and returns no output that shares memory with an input.**
The framework passes the values it holds without a copy, such as another record's output that a request references.
A binding that modified one, or returned a view of one, would change another record's value.
`FORWARD` in `toy_sans.py` returns `curve['data']['Q', 0].copy()` for this reason.

## 3. Stages: compute the fixed part once

When a user tunes `bins`, the runs stay the same.
The backend then *stages* the binding: it calls `stage` with the values that stay fixed and the names of the blanks, the parameters left open, and calls the returned callable once per value of the blanks.
(The client's side of this is called a *stage* too: a template the backend keeps for a client, see [client-authors.ipynb](client-authors.ipynb).)
`PipelineBinding` cuts the pipeline at the keys of the blanks with `sciline.Stage`, and computes what does not depend on them once:

In [8]:
fixed = {key: values[key] for key in ('sample', 'can')}
call = single.stage(fixed, ('reduction',))
toy_sans.LOADED.clear()
for bins in (1, 2, 4):
    print(bins, call(reduction={'bins': bins})['iofq'].values)
print('runs loaded:', len(toy_sans.LOADED))

1 [1.6]
2 [1.2 0.4]
4 [0.7 0.5 0.3 0.1]
runs loaded: 2


Two loads for three calls: the sample run and the can run, once each.
A function binding stages nothing; the backend calls it with every value each time.

**Promise: a call through a stage gives the outputs of the plain request** (the *caching* symmetry).
The framework records a call through a stage as its plain request, so the record must not depend on how it was computed.
`ess.spec.testing` checks it by computing both, and also checks that no output shares memory with an input:

In [9]:
from ess.spec.testing import check_caching

check_caching(single, fixed, [{'reduction': {'bins': n}} for n in (2, 1, 2)])

## 4. The spec of several runs: tables

Many reductions combine runs: here, several sample runs and several can runs, summed.
A *table* is a list of rows, and a row is a flat model too.
The row is what the reduction repeats per run, so the same `Run` is a group of the single-run spec and a row of the multi-run spec:

In [10]:
class MultiIofQParams(BaseModel):
    sample_runs: list[Run] = Field(min_length=1)
    can_runs: list[Run] = Field(min_length=1)
    reduction: Reduction = Field(default_factory=Reduction)


IOFQ_MULTI = WorkflowSpec(
    name='toy-sans-iofq-multi',
    version=1,
    title='I(Q) of summed runs',
    description='I(Q) of the summed sample runs, minus that of the summed can runs.',
    params=MultiIofQParams,
    outputs=IofQOutputs,
)

Rules of tables:

- **A row holds what belongs to one run**, such as a sample run and its own transmission run. Rows of different tables are independent; pairing runs into rows is the app's job.
- **Structure below a row is the binding's**, such as detector banks. The framework sees rows only.
- **The multi-run spec declares the same outputs model as the single-run spec**, and with one row per table gives the same outputs (the *one-row* symmetry).

The binding of the multi-run spec takes the same pipeline of one run.
It maps the same keys as the single-run binding, with each row of a table in place of a group: `'sample_runs.file'` is the field `file` of each row.
It also names each table's *accumulated keys*: the values of one run that add up over runs, here the numerator and the denominator.

In [11]:
from ess.spec.pipeline import AccumulatingPipelineBinding

multi_keys = {
    'sample_runs.file': RunData[SampleRun],
    'can_runs.file': RunData[CanRun],
    'reduction.bins': Bins,
}
summed = {
    'sample_runs': (Numerator[SampleRun], Denominator[SampleRun]),
    'can_runs': (Numerator[CanRun], Denominator[CanRun]),
}
multi = AccumulatingPipelineBinding(
    pipeline, params=multi_keys, outputs={'iofq': IofQ}, accumulate=summed
)
tables = {
    'sample_runs': [{'file': SAMPLE_RUNS[101]}, {'file': SAMPLE_RUNS[102]}],
    'can_runs': [{'file': CAN_RUNS[110]}],
}
multi.stage({**tables, 'reduction': {'bins': 2}}, ())()

{'iofq': <scipp.Variable> (Q: 2)    float64  [dimensionless]  [0.8, 0.133333]}

A plain request through it computes each row's accumulated keys, sums them, and computes the outputs from the sums.
Summing numerators and denominators apart, and dividing once, is what makes a sum of runs correct; a sum of normalized curves would not be.

A table with no rows stands for an optional run left out.
The binding then sets the keys of the table's fields to `None` and computes one row, as `PipelineBinding` does for a group that is `None`, so `can_runs=[]` gives what `can=None` gives.
In this toy both kinds of run are required (`min_length=1`), so the backend never passes `None`: it refuses to read a state whose plain request has an empty table.
`check_arrival_and_order` below does compute such states, and skips those whose plain request raises.

**Promise: one row per table gives the outputs of the single-run spec** (the *one-row* symmetry):

In [12]:
from ess.spec.testing import check_one_row

check_one_row(
    single,
    multi,
    values,
    {'sample_runs': [values['sample']], 'can_runs': [values['can']], 'reduction': {'bins': 2}},
)

## 5. Accumulating: a held state

When runs arrive one at a time, a user wants the sum so far after each, without reducing the earlier runs again.
The framework's *accumulator* takes one *push* at a time, each a row for one or more tables (see [client-authors.ipynb](client-authors.ipynb)).
A binding that can add a row to what it holds provides `held_state(fixed)`: an object with `push(rows)` and `outputs()`.

The protocol is small enough to write by hand.
This held state keeps a numerator and a denominator per table, the same sums `multi` names:

In [13]:
import numpy as np
import scipp as sc


class HeldSums:
    """Numerator and denominator of each table, summed over the rows pushed."""

    def __init__(self, bins: int) -> None:
        self._bins = bins
        self._sums: dict[str, tuple[np.ndarray, float]] = {}

    def push(self, rows: dict) -> None:
        for table, row in rows.items():
            counts = np.asarray(row['file']['counts'])
            numerator = counts.reshape(self._bins, -1).sum(axis=1)
            denominator = row['file']['monitor']
            if table in self._sums:
                total, norm = self._sums[table]
                total += numerator  # in place: the held state is the binding's own
                self._sums[table] = (total, norm + denominator)
            else:
                self._sums[table] = (numerator, denominator)

    def outputs(self) -> dict:
        sample, sample_norm = self._sums['sample_runs']
        can, can_norm = self._sums['can_runs']
        curve = sample / sample_norm - can / can_norm
        return {'iofq': sc.array(dims=['Q'], values=curve, unit='dimensionless')}


class HandBinding:
    def held_state(self, fixed: dict) -> HeldSums:
        return HeldSums(fixed['reduction']['bins'])

    def stage(self, fixed: dict, blanks: tuple):
        def call(**values):  # a plain request pushes every row into a new held state
            given = {**fixed, **values}
            held = self.held_state(given)
            for table in ('sample_runs', 'can_runs'):
                for row in given[table]:
                    held.push({table: row})
            return held.outputs()

        return call


hand = HandBinding()
held = hand.held_state({'reduction': {'bins': 2}})
held.push({'sample_runs': tables['sample_runs'][0], 'can_runs': tables['can_runs'][0]})
held.push({'sample_runs': tables['sample_runs'][1]})
held.outputs()

{'iofq': <scipp.Variable> (Q: 2)    float64  [dimensionless]  [0.8, 0.133333]}

The binding's author promises three things, which the backend does not check:

- **After rows are pushed, `outputs()` gives what the plain request over those rows gives**, whatever the order of pushes into different tables (the *arrival* and *order* symmetries). So each sum may depend on the fixed values and on one table only.
- **`push` may change the held state in place, never the rows passed to it.** A row may be another record's output.
- **Validators that read several values, such as the params model's own, change no value.** The held state gets each fixed value checked by its own field alone. The backend runs the params model on the plain request of a state when the state is read, but does not check that the held state got the same values.

In return, `outputs()` may share memory with what the held state holds: the backend is done with the outputs of a state before it pushes the next rows.

`check_arrival_and_order` pushes the rows in several orders and compares each state with its plain request, skipping states whose plain request raises, which the backend would refuse to read.
`HandBinding` computes a plain request by pushing every row into a new held state, so arrival holds by construction; the check still catches a held state that depends on the order of pushes into different tables, or that modifies its rows.

In [14]:
from ess.spec.testing import check_arrival_and_order

check_arrival_and_order(hand, {'reduction': {'bins': 2}}, tables)
check_arrival_and_order(multi, {'reduction': {'bins': 2}}, tables)

`AccumulatingPipelineBinding` provides the held state from the pipeline: it computes the accumulated keys of each pushed row and adds them to the sums.
A push loads one run:

In [15]:
held = multi.held_state({'reduction': {'bins': 2}})
toy_sans.LOADED.clear()
for row in tables['sample_runs']:
    held.push({'sample_runs': row})
held.push({'can_runs': tables['can_runs'][0]})
print('runs loaded:', len(toy_sans.LOADED))
held.outputs()

runs loaded: 3


{'iofq': <scipp.Variable> (Q: 2)    float64  [dimensionless]  [0.8, 0.133333]}

A sum that depends on two tables would break the order symmetry: the can's sum would change with the sample rows pushed before it.
esssans has this case, since its pixel masks come from the sample run.
`AccumulatingPipelineBinding` refuses such a pipeline when it is made, so a package fails at import, not at a user's push:

In [16]:
from typing import NewType

Mask = NewType('Mask', sc.Variable)


def sample_mask(counts: Counts[SampleRun]) -> Mask:
    return Mask(counts == sc.scalar(0.0, unit='counts'))


def masked_numerator(counts: Counts[RunType], mask: Mask, bins: Bins) -> Numerator[RunType]:
    kept = sc.where(mask, sc.zeros_like(counts), counts)
    binned = kept.values.reshape(bins, -1).sum(axis=1)
    return Numerator[RunType](sc.array(dims=['Q'], values=binned, unit='counts'))


masked = pipeline.copy()
masked.insert(sample_mask)
masked.insert(masked_numerator)
try:
    AccumulatingPipelineBinding(
        masked, params=multi_keys, outputs={'iofq': IofQ}, accumulate=summed
    )
except ValueError as error:
    print(error)

the accumulated key toy_sans.Numerator[toy_sans.CanRun] of can_runs depends on the rows of ['can_runs', 'sample_runs'], not of can_runs alone


A binding without `held_state` works in an accumulator too.
The backend then keeps the rows and computes the plain request over all of them for each state that is read.
The outputs are the same; only the cost differs.

## 6. What the framework gives the binding

The binding may rely on this:

| Call | When, and with what |
|---|---|
| `stage(fixed, blanks)` | once per plain request, once per stage of a client, and, for a binding without `held_state`, once per state that is read; every value but the blanks, with data read and defaults filled in; possibly from several threads at once |
| the callable it returns | once per request through the stage; possibly from several threads at once |
| `held_state(fixed)` | once per accumulator; every value but the tables it fills |
| `push(rows)` | once per push, in the order logged; one thread at a time; never while the outputs of the state before it are in use |
| `outputs()` | once per state that someone reads; one thread at a time |

[framework-promises.ipynb](framework-promises.ipynb) traces these calls for one accumulator.

## Summary: what the author of a workflow promises

| Promise | Checked by |
|---|---|
| defaults live in the params model only | nobody |
| a check of several values is a model validator, and changes no value | nobody |
| a new version when the params or outputs models change | nobody |
| the binding returns no output that shares memory with an input | `check_one_row`, `check_caching` (numpy-backed values) |
| the binding modifies no input | nobody, except for the rows a push gets (`check_arrival_and_order`) |
| a call through a stage gives the outputs of the plain request | `check_caching` |
| the multi-run spec with one row per table gives the single-run spec's outputs | `check_one_row` |
| after n pushes, the held state's outputs are the plain request's over their rows, in any order across tables | `check_arrival_and_order`; for the order, also `AccumulatingPipelineBinding` when it is made |
| `push` modifies no row | `check_arrival_and_order` |
| structure below a row belongs to the binding | nobody |

A package registers its bindings through the entry-point group `ess.dispatch.workflows` (decided, not implemented).
Until then, a notebook or app passes them to `local(proposal=..., datasets=..., bind={IOFQ: single, IOFQ_MULTI: multi})`.